# 🌸 Iris Flower Classification with Machine Learning
---
**Goal:** Train a model that learns from sepal/petal measurements and correctly classifies an Iris flower into one of three species:
- 🟢 *Iris-setosa*
- 🔵 *Iris-versicolor*
- 🔴 *Iris-virginica*

**Pipeline:** Load → Explore → Visualize → Preprocess → Train → Evaluate → Predict

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection    import train_test_split, cross_val_score
from sklearn.preprocessing      import LabelEncoder, StandardScaler
from sklearn.linear_model       import LogisticRegression
from sklearn.neighbors          import KNeighborsClassifier
from sklearn.tree               import DecisionTreeClassifier, plot_tree
from sklearn.ensemble           import RandomForestClassifier
from sklearn.svm                import SVC
from sklearn.metrics            import (accuracy_score, classification_report,
                                         confusion_matrix, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110
print('✅ Libraries ready!')

## Step 2 — Load the Dataset

In [ ]:
# ── Upload your Iris.csv in Colab ────────────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()          # click 'Choose Files' and select Iris.csv
# df = pd.read_csv('Iris.csv')

# ── OR load directly from URL (no upload needed) ─────────────────────────────
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/iris.csv'
df  = pd.read_csv(url)
df.columns = ['Id','SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm','Species']

print(f'Dataset shape : {df.shape}')
print(f'Species found : {df.Species.unique().tolist()}')
df.head(10)

## Step 3 — Explore the Dataset

In [ ]:
print('=== Data Types & Nulls ===')
print(df.dtypes)
print('\nNull values:\n', df.isnull().sum())

In [ ]:
print('=== Class Distribution ===')
print(df['Species'].value_counts())
print('\n=== Statistical Summary ===')
df.describe().T.style.format('{:.2f}').background_gradient(cmap='Blues')

In [ ]:
print('=== Mean measurements per species ===')
features = ['SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm']
df.groupby('Species')[features].mean().round(2).style.background_gradient(cmap='YlGn')

## Step 4 — Data Visualization

In [ ]:
# ── 4a. Species count ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
colors = ['#66c2a5', '#fc8d62', '#8da0cb']
df['Species'].value_counts().plot(kind='bar', ax=ax, color=colors, edgecolor='black', width=0.6)
ax.set_title('Species Count', fontsize=13, fontweight='bold')
ax.set_xlabel('Species') ; ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()+0.3), ha='center')
plt.tight_layout() ; plt.show()

In [ ]:
# ── 4b. Boxplots – one per feature ───────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Feature Distributions by Species', fontsize=14, fontweight='bold')
for ax, feat in zip(axes.flatten(), features):
    sns.boxplot(data=df, x='Species', y=feat, ax=ax, palette='Set2')
    ax.set_title(feat, fontweight='bold') ; ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=12)
plt.tight_layout() ; plt.show()

In [ ]:
# ── 4c. Scatter — Sepal vs Petal (key separator) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Sepal vs Petal — Key Separators', fontsize=13, fontweight='bold')

palette = {'Iris-setosa':'#66c2a5', 'Iris-versicolor':'#fc8d62', 'Iris-virginica':'#8da0cb'}

sns.scatterplot(data=df, x='SepalLengthCm', y='SepalWidthCm',
                hue='Species', palette=palette, ax=axes[0], s=70, edgecolor='k', linewidth=0.4)
axes[0].set_title('Sepal Length vs Width')

sns.scatterplot(data=df, x='PetalLengthCm', y='PetalWidthCm',
                hue='Species', palette=palette, ax=axes[1], s=70, edgecolor='k', linewidth=0.4)
axes[1].set_title('Petal Length vs Width')

plt.tight_layout() ; plt.show()

In [ ]:
# ── 4d. Correlation heatmap ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
corr = df[features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout() ; plt.show()

print('💡 Insight: Petal length & width are highly correlated (>0.96).')
print('   Petal features are the strongest classifiers for species separation.')

In [ ]:
# ── 4e. Full pairplot ─────────────────────────────────────────────────────────
g = sns.pairplot(df.drop('Id', axis=1), hue='Species', palette='Set2',
                 diag_kind='kde', plot_kws={'alpha':0.6, 's':40, 'edgecolor':'k', 'linewidth':0.3})
g.fig.suptitle('Pairplot — All Features vs All Features', y=1.02, fontsize=13, fontweight='bold')
plt.show()

## Step 5 — Preprocessing

In [ ]:
# Drop non-feature column
df_ml = df.drop('Id', axis=1).copy()

# Encode labels:  setosa=0, versicolor=1, virginica=2
le = LabelEncoder()
df_ml['Target'] = le.fit_transform(df_ml['Species'])
print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

X = df_ml[features]
y = df_ml['Target']

# 80 / 20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

# Feature scaling
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'\nTraining samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')
print(f'Features         : {X.shape[1]}')

## Step 6 — Train & Compare Models

In [ ]:
models = {
    'Logistic Regression' : LogisticRegression(max_iter=300, random_state=42),
    'K-Nearest Neighbors' : KNeighborsClassifier(n_neighbors=5),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=150, random_state=42),
    'Support Vector Machine': SVC(kernel='rbf', C=10, gamma='scale', probability=True),
}

results = []
trained = {}

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    trained[name] = model

    train_acc = accuracy_score(y_train, model.predict(X_train_sc))
    test_acc  = accuracy_score(y_test,  model.predict(X_test_sc))
    cv_scores = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='accuracy')

    results.append({'Model'     : name,
                    'Train Acc' : round(train_acc,        4),
                    'Test Acc'  : round(test_acc,         4),
                    'CV Mean'   : round(cv_scores.mean(), 4),
                    'CV Std'    : round(cv_scores.std(),  4)})
    print(f'{name:<26}  Train: {train_acc:.2%}  Test: {test_acc:.2%}  CV: {cv_scores.mean():.2%}±{cv_scores.std():.2%}')

results_df = pd.DataFrame(results).sort_values('Test Acc', ascending=False).reset_index(drop=True)
print('\n🏆 Best Model:', results_df.iloc[0]['Model'])

In [ ]:
# ── Model comparison bar chart ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
x     = np.arange(len(results_df))
width = 0.28

b1 = ax.bar(x - width,   results_df['Train Acc'], width, label='Train Acc', color='#4e9af1', edgecolor='black')
b2 = ax.bar(x,           results_df['Test Acc'],  width, label='Test Acc',  color='#f97316', edgecolor='black')
b3 = ax.bar(x + width,   results_df['CV Mean'],   width, label='CV Mean',   color='#22c55e', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right', fontsize=10)
ax.set_ylabel('Accuracy')
ax.set_ylim(0.88, 1.03)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend()
ax.axhline(1.0, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)

for bar in [*b1, *b2, *b3]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

## Step 7 — Evaluate Best Model in Detail

In [ ]:
# Identify best model automatically
best_name  = results_df.iloc[0]['Model']
best_model = trained[best_name]
y_pred     = best_model.predict(X_test_sc)

print(f'🏆 Best Model : {best_name}')
print(f'   Test Accuracy : {accuracy_score(y_test, y_pred):.2%}\n')
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=12, fontweight='bold')
plt.tight_layout() ; plt.show()

print('Diagonal = correct predictions | Off-diagonal = misclassifications')

In [ ]:
# ── Feature importance (Random Forest) ───────────────────────────────────────
rf = trained['Random Forest']
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#8da0cb','#8da0cb','#fc8d62','#fc8d62']
importances.plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='black')
ax.set_title('Feature Importance — Random Forest', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
for i, (val, name) in enumerate(zip(importances, importances.index)):
    ax.text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=10)
plt.tight_layout() ; plt.show()

print('💡 Petal features dominate — they are the strongest classifiers.')

In [ ]:
# ── Decision Tree: visual explanation of the rules ───────────────────────────
dt = trained['Decision Tree']
fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(dt,
          feature_names=features,
          class_names=le.classes_,
          filled=True, rounded=True,
          fontsize=9, ax=ax,
          impurity=False, proportion=False)
ax.set_title('Decision Tree — Classification Rules', fontsize=13, fontweight='bold')
plt.tight_layout() ; plt.show()

## Step 8 — Predict on New Flower Measurements 🌸

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  ✏️  Change these four measurements to classify any iris flower!
# ──────────────────────────────────────────────────────────────────────────────
sepal_length = 5.9
sepal_width  = 3.0
petal_length = 5.1
petal_width  = 1.8
# ──────────────────────────────────────────────────────────────────────────────

sample = pd.DataFrame([[sepal_length, sepal_width, petal_length, petal_width]],
                       columns=features)
sample_sc    = scaler.transform(sample)

pred_class   = best_model.predict(sample_sc)[0]
pred_label   = le.inverse_transform([pred_class])[0]
pred_proba   = best_model.predict_proba(sample_sc)[0]

emoji = {'Iris-setosa':'🟢', 'Iris-versicolor':'🔵', 'Iris-virginica':'🔴'}

print('╔══════════════════════════════════════════╗')
print('║         IRIS FLOWER PREDICTION           ║')
print('╠══════════════════════════════════════════╣')
print(f'║  Sepal Length : {sepal_length} cm                    ║')
print(f'║  Sepal Width  : {sepal_width} cm                    ║')
print(f'║  Petal Length : {petal_length} cm                    ║')
print(f'║  Petal Width  : {petal_width} cm                    ║')
print('╠══════════════════════════════════════════╣')
print(f'║  Predicted    : {emoji[pred_label]} {pred_label:<25}║')
print('╠══════════════════════════════════════════╣')
print('║  Confidence Scores:                      ║')
for cls, prob in zip(le.classes_, pred_proba):
    bar  = '█' * int(prob * 20)
    note = ' ← predicted' if cls == pred_label else ''
    print(f'║  {emoji[cls]} {cls:<18} {prob:.2%}  {bar:<20}{note}')
print('╚══════════════════════════════════════════╝')

In [ ]:
# ── Confidence bar chart ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 3.5))
bar_colors = ['#66c2a5','#fc8d62','#8da0cb']
bars = ax.barh(le.classes_, pred_proba, color=bar_colors, edgecolor='black', height=0.5)
ax.set_xlim(0, 1.1)
ax.set_xlabel('Probability')
ax.set_title(f'Prediction Confidence  →  {pred_label}', fontsize=12, fontweight='bold')
for bar, val in zip(bars, pred_proba):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2%}', va='center', fontsize=11)
plt.tight_layout() ; plt.show()

## Step 9 — Final Summary & Key Insights

In [ ]:
print('╔══════════════════════════════════════════════════════╗')
print('║         IRIS CLASSIFICATION — FINAL SUMMARY         ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Dataset     : 150 samples | 3 classes | 4 features  ║')
print(f'║  Train/Test  : 120 / 30 (stratified split)           ║')
print('╠══════════════════════════════════════════════════════╣')
print('║  Model Leaderboard:                                  ║')
for _, row in results_df.iterrows():
    marker = ' 🏆' if row['Model'] == results_df.iloc[0]['Model'] else '   '
    print(f"║  {marker} {row['Model']:<28} Test={row['Test Acc']:.2%}   ║")
print('╠══════════════════════════════════════════════════════╣')
print('║  Key Insights:                                       ║')
print('║  • Iris-setosa is perfectly linearly separable       ║')
print('║  • Petal features are the strongest classifiers      ║')
print('║  • versicolor & virginica overlap slightly           ║')
print('║  • All models achieve >95% accuracy on this dataset  ║')
print('╚══════════════════════════════════════════════════════╝')